In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import glob
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import imageio
from xarray.coders import CFDatetimeCoder

# —— 通用子程序 —— #

def compute_edges(coord):
    d = np.diff(coord) / 2
    return np.concatenate([[coord[0] - d[0]], coord[:-1] + d, [coord[-1] + d[-1]]])

def pressure_to_height(plev, p0=1e5, H=7000.0):
    p = plev.values
    if np.nanmax(p) < 2000:
        p *= 100.0
    z = -H * np.log(p / p0) / 1000.0
    return xr.DataArray(z, coords={'plev': plev}, dims=['plev'])

def load_hindcast(pattern, start, end):
    coder = CFDatetimeCoder(use_cftime=True)
    files = sorted(glob.glob(pattern))
    dss = []
    for f in files:
        ds = xr.open_dataset(f, decode_times=coder)
        sel = ds.sel(time=slice(start, end))
        if sel.time.size > 0:
            dss.append(sel)
    if not dss:
        raise RuntimeError(f"No hindcast data for pattern {pattern}")
    return xr.concat(dss, dim='member').mean(dim='member')

def load_control(path, start, end):
    coder = CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(path, decode_times=coder)
    sel = ds.sel(time=slice(start, end))
    if sel.time.size == 0:
        raise RuntimeError(f"No control data for slice {start}–{end}")
    return sel

# —— 计算 CPT 曲线 —— #

def compute_cpt_heights(T2d, height):
    """
    T2d: DataArray (plev, lat)
    height: DataArray of same plev → height (km)
    返回 1d 数组 (lat,)
    """
    t_vals = T2d.values
    idx = np.nanargmin(t_vals, axis=0)        # 每个纬度最小温度所在层
    h = height.values
    return h[idx]

# —— 季节平均拼接图 —— #
#注意这里先计算了温度平均然后再根据冷点绘制的冷点曲线
def plot_spliced_seasonal_average(datasets, lat, height, output_file):
    lat_e, h_e = compute_edges(lat), compute_edges(height)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height)
    tags = list(datasets.keys())

    fig, axes = plt.subplots(2, 3, figsize=(15, 10),
                             sharex=True, sharey=True)
    axes = axes.flatten()

    for i, tag in enumerate(tags):
        ds = datasets[tag]
        # O3 与 U 的季节平均
        o3 = ds['O3'].mean(dim=['time','lon'])
        u  = ds['U'].mean(dim=['time','lon'])
        # T 的季节平均用于 CPT
        Tm = ds['T'].mean(dim=['time','lon'])
        cpt = compute_cpt_heights(Tm, height)

        ax = axes[i]
        pcm = ax.pcolormesh(LAT_e, HT_e, o3.values,
                            shading='auto', cmap='rainbow')
        ct  = ax.contour(LATc, HTc, u.values,
                         levels=[20,30,40,50],
                         colors='white', linewidths=1)
        ax.clabel(ct, fmt='%d', inline=True, fontsize=8)

        # 叠加 CPT 曲线
        ax.plot(lat, cpt, color='k', linewidth=2)

        ax.set_title(f'({chr(97+i)}) {tag}')
        ax.set_xlabel('Latitude (°)')
        ax.set_ylabel('Height (km)')
        ax.set_ylim(0, 30)

    fig.delaxes(axes[-1])
    cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    fig.colorbar(pcm, cax=cax, orientation='vertical',
                 label='O3 mixing ratio (mol/mol)')
    plt.tight_layout(rect=[0, 0, 0.9, 1])
    fig.savefig(output_file)
    plt.close(fig)

# —— 单场景 GIF （含等值线＋CPT） —— #

def create_o3_gif(ds, lat, height, frames_dir, gif_file, duration=0.5):
    os.makedirs(frames_dir, exist_ok=True)
    lat_e, h_e = compute_edges(lat), compute_edges(height)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height)

    writer = imageio.get_writer(gif_file, mode='I', duration=duration)
    for i, t in enumerate(ds.time.values):
        o3 = ds['O3'].isel(time=i).mean(dim='lon')
        u  = ds['U'].isel(time=i).mean(dim='lon')
        Tm = ds['T'].isel(time=i).mean(dim='lon')
        cpt = compute_cpt_heights(Tm, height)

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.pcolormesh(LAT_e, HT_e, o3.values,
                      shading='auto', cmap='rainbow')
        ct = ax.contour(LATc, HTc, u.values,
                        levels=[20,30,40,50],
                        colors='white', linewidths=1)
        ax.clabel(ct, fmt='%d', inline=True, fontsize=8)

        ax.plot(lat, cpt, color='k', linewidth=2)

        ax.set_title(str(t))
        ax.set_xlabel('Latitude (°)')
        ax.set_ylabel('Height (km)')
        ax.set_ylim(0, 30)

        frame = os.path.join(frames_dir,
                             f'frame_{i:03d}.png')
        fig.savefig(frame)
        plt.close(fig)
        writer.append_data(imageio.imread(frame))
    writer.close()

# —— 合成 GIF （含CPT） —— #

def create_combined_gif(datasets, lat, height,
                        frames_dir, combo_file,
                        duration=0.5):
    os.makedirs(frames_dir, exist_ok=True)
    lat_e, h_e = compute_edges(lat), compute_edges(height)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height)

    times = next(iter(datasets.values())).time.values
    writer = imageio.get_writer(combo_file, mode='I', duration=duration)

    for i, t in enumerate(times):
        fig, axes = plt.subplots(2, 3, figsize=(15, 10),
                                 sharex=True, sharey=True)
        axes = axes.flatten()
        for j, tag in enumerate(datasets):
            ds = datasets[tag]
            o3_ppmv = ds['O3'].isel(time=i).mean(dim='lon') * 1e6
            u2d     = ds['U'].isel(time=i).mean(dim='lon')
            Tm      = ds['T'].isel(time=i).mean(dim='lon')
            cpt     = compute_cpt_heights(Tm, height)

            ax = axes[j]
            pcm = ax.pcolormesh(LAT_e, HT_e, o3_ppmv.values,
                                shading='auto', cmap='rainbow')
            ct  = ax.contour(LATc, HTc, u2d.values,
                             levels=[20,30,40,50],
                             colors='white', linewidths=1)
            ax.clabel(ct, fmt='%d', inline=True, fontsize=8)

            ax.plot(lat, cpt, color='k', linewidth=2)

            ax.set_title(f'({chr(97+j)}) {tag}')
            ax.set_xlabel('Latitude (°)')
            ax.set_ylabel('Height (km)')
            ax.set_ylim(0, 30)

        fig.delaxes(axes[-1])
        cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        fig.colorbar(pcm, cax=cax, orientation='vertical',
                     label='O₃ mixing ratio (ppmv)')
        plt.tight_layout(rect=[0, 0, 0.9, 1])
        frame = os.path.join(frames_dir,
                             f'combo_{i:03d}.png')
        fig.savefig(frame)
        plt.close(fig)
        writer.append_data(imageio.imread(frame))
    writer.close()

# —— 差值 GIF （含CPT） —— #

def create_spliced_difference_gif(datasets, lat, height,
                                  frames_dir, gif_file,
                                  duration=0.5):
    os.makedirs(frames_dir, exist_ok=True)
    lat_e, h_e = compute_edges(lat), compute_edges(height)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height)
    exp_tags = [t for t in datasets if t != 'Reference']
    times    = datasets[exp_tags[0]].time.values
    writer   = imageio.get_writer(gif_file, mode='I', duration=duration)

    for i, t in enumerate(times):
        fig, axes = plt.subplots(2, 2, figsize=(12, 8),
                                 sharex=True, sharey=True)
        axes = axes.flatten()
        for j, tag in enumerate(exp_tags):
            ds      = datasets[tag]
            diff2d  = (ds['O3'].isel(time=i)
                       - datasets['Reference']['O3'].isel(time=i))
            diff2d  = diff2d.mean(dim='lon') * 1e6
            u2d     = ds['U'].isel(time=i).mean(dim='lon')
            Tm      = ds['T'].isel(time=i).mean(dim='lon')
            cpt     = compute_cpt_heights(Tm, height)

            ax = axes[j]
            pcm = ax.pcolormesh(LAT_e, HT_e, diff2d.values,
                                shading='auto', cmap='RdBu_r',
                                vmin=-1, vmax=1)
            ct  = ax.contour(LATc, HTc, u2d.values,
                             levels=[20,30,40,50],
                             colors='white', linewidths=1)
            ax.clabel(ct, fmt='%d', inline=True, fontsize=8)

            ax.plot(lat, cpt, color='k', linewidth=2)

            ax.set_title(f'({chr(97+j)}) {tag} – Reference')
            ax.set_xlabel('Latitude (°)')
            ax.set_ylabel('Height (km)')
            ax.set_ylim(0, 30)

        plt.tight_layout(rect=[0, 0, 0.9, 1])
        cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        fig.colorbar(pcm, cax=cax, orientation='vertical',
                     label='ΔO₃ (ppmv)')
        frame = os.path.join(frames_dir,
                             f'diff_{i:03d}.png')
        fig.savefig(frame)
        plt.close(fig)
        writer.append_data(imageio.imread(frame))
    writer.close()

# —— 执行流程 —— #

year = '0008'
start, end = "0008-03-01", "0008-05-31"
PARENT = "/home/weiji/restart_exam/plots/O3_mixingratio"
os.makedirs(PARENT, exist_ok=True)

patterns = [
    (f"{year}_Feb_couple",
     f"/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
     f"BWCN.e122.f19_g16.002_{year}/Feb/"
     f"BWCN.e122.f19_g16.002.{year}-02.*.nc"),
    (f"{year}_Mar_couple",
     f"/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
     f"BWCN.e122.f19_g16.002_{year}/Mar/"
     f"BWCN.e122.f19_g16.002.{year}-03.*.nc"),
    (f"{year}_Feb_nocouple",
     f"/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/"
     f"{year}-02/*.nc"),
    (f"{year}_Mar_nocouple",
     f"/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/"
     f"{year}-03/*.nc"),
]
control_file = (
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.002/"
    "BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc"
)

datasets = { tag: load_hindcast(p, start, end) for tag,p in patterns }
datasets['Reference'] = load_control(control_file, start, end)

example = next(iter(datasets.values()))
lat     = example.lat.values
plev    = example.plev
height  = pressure_to_height(plev)

# 季节平均拼接图
plot_spliced_seasonal_average(
    datasets, lat, height,
    os.path.join(PARENT, "MAM_average_spliced.png")
)

# 单场景 GIF
for tag, ds in datasets.items():
    scen_dir = os.path.join(PARENT, tag)
    frames   = os.path.join(scen_dir, "frames")
    gif_file = os.path.join(scen_dir, f"O3_{tag}.gif")
    os.makedirs(scen_dir, exist_ok=True)
    create_o3_gif(ds, lat, height, frames, gif_file)

# 合成 GIF
combo_frames = os.path.join(PARENT, "combined_frames")
combo_gif    = os.path.join(PARENT, "O3_combined.gif")
create_combined_gif(datasets, lat, height,
                    combo_frames, combo_gif, duration=0.5)

# 差值 GIF
diff_frames = os.path.join(PARENT, "diff_frames")
diff_gif    = os.path.join(PARENT, "O3_diff_combined.gif")
create_spliced_difference_gif(datasets, lat, height,
                              diff_frames, diff_gif, duration=0.5)


In [ ]:
def create_spliced_difference_gif(datasets, lat, height,
                                  frames_dir, gif_file,
                                  duration=0.5):
    """
    为 4 个 hindcast 场景生成 (O3_exp − O3_ref) 差值的 2×2 面板 GIF，
    每帧叠加对应场景的 U 等值线 (20/30/40/50 m/s)，
    差值单位转换为 ppmv（×1e6），色标范围固定为 [-1, 1] ppmv。
    """
    os.makedirs(frames_dir, exist_ok=True)
    # 构造网格边界
    lat_e, h_e = compute_edges(lat), compute_edges(height)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height)

    # 只保留四个 hindcast 场景
    exp_tags = [t for t in datasets if t != 'Reference']
    times    = datasets[exp_tags[0]].time.values

    writer = imageio.get_writer(gif_file, mode='I', duration=duration)
    for i, t in enumerate(times):
        fig, axes = plt.subplots(2, 2, figsize=(12, 8),
                                 sharex=True, sharey=True)
        axes = axes.flatten()

        for j, tag in enumerate(exp_tags):
            ds     = datasets[tag]
            o3_exp = ds['O3'].isel(time=i)
            o3_ref = datasets['Reference']['O3'].isel(time=i)
            # 差值并转换为 ppmv
            diff2d = (o3_exp - o3_ref).mean(dim='lon') * 1e6  
            u2d    = ds['U'].isel(time=i).mean(dim='lon')

            ax = axes[j]
            pcm = ax.pcolormesh(
                LAT_e, HT_e, diff2d.values,
                shading='auto', cmap='RdBu_r',
                vmin=-1, vmax=1
            )
            ct = ax.contour(
                LATc, HTc, u2d.values,
                levels=[20, 30, 40, 50],
                colors='white', linewidths=1
            )
            ax.clabel(ct, fmt='%d', inline=True, fontsize=8)
            ax.set_title(f'({chr(97+j)}) {tag} – Reference')
            ax.set_ylim(0, 30)
            ax.set_xlabel('Latitude (°)')
            ax.set_ylabel('Height (km)')

        plt.tight_layout(rect=[0, 0, 0.9, 1])
        cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        fig.colorbar(pcm, cax=cax, orientation='vertical',
                     label='ΔO3 (ppmv)')

        frame = os.path.join(frames_dir, f'diff_{i:03d}.png')
        fig.savefig(frame)
        plt.close(fig)
        writer.append_data(imageio.imread(frame))
    writer.close()




# 差值 GIF
diff_frames = os.path.join(PARENT, 'diff_frames')
diff_gif    = os.path.join(PARENT, 'O3_diff_combined.gif')
create_spliced_difference_gif(
    datasets, lat, height,
    diff_frames, diff_gif,
    duration=0.5
)



In [ ]:
# —— Cold Point Tropopause (CPT) 曲线绘制 —— #

# 1. 选取 Feb_couple 第一个时刻数据
tag = f"{year}_Feb_couple"
ds0 = datasets[tag].isel(time=0)

# 2. 计算 lon 平均的 O3 (ppmv)、U 和 T
O3_ppmv = ds0['O3'].mean(dim='lon')  * 1e6     # mol/mol → ppmv
U_lon   = ds0['U'].mean(dim='lon')
T_lon   = ds0['T'].mean(dim='lon')

# 3. 找到每个纬度上的温度最小层（Cold Point Tropopause）
#    T_lon 维度顺序 (plev, lat)
t_vals = T_lon.values                # shape (nplev, nlat)
cpt_idx = np.nanargmin(t_vals, axis=0)   # 每列最小值的索引
# height 是 pressure_to_height 计算出的 DataArray (plev)
h_vals = height.values               # shape (nplev,)
cpt_heights = h_vals[cpt_idx]        # shape (nlat,)

# 4. 构造绘图网格
lat_e, h_e = compute_edges(lat), compute_edges(height)
LAT_e, HT_e = np.meshgrid(lat_e, h_e)
LATc, HTc   = np.meshgrid(lat, height)

# 5. 绘制 Lat–Altitude 背景图：O3_ppmv + U 等值线 + CPT 曲线
fig, ax = plt.subplots(figsize=(10, 6))
# O3 背景填色
pcm = ax.pcolormesh(
    LAT_e, HT_e, O3_ppmv.values,
    shading='auto', cmap='rainbow'
)
# U 等值线
ct = ax.contour(
    LATc, HTc, U_lon.values,
    levels=[20, 30, 40, 50],
    colors='white', linewidths=1
)
ax.clabel(ct, fmt='%d', inline=True, fontsize=8)
# CPT 曲线
ax.plot(lat, cpt_heights,
        color='k', linewidth=2, label='Cold-Point Tropopause')
ax.legend(loc='upper right')

# 6. 坐标、色标和保存
ax.set_xlabel('Latitude (°)')
ax.set_ylabel('Height (km)')
ax.set_ylim(0, 30)
cbar = fig.colorbar(pcm, ax=ax, orientation='vertical', pad=0.02)
cbar.set_label('O₃ mixing ratio (ppmv)')
fig.tight_layout()
fig.savefig(os.path.join(PARENT, f"{tag}_CPT.png"))
plt.show(fig)
